# Passport biodata-page detection from multi-angle OCR

This notebook loads `ocr_8_angles.csv`, searches every OCR angle, and returns one row per image with the matched regex pattern names and a `YES`/`NO` verdict.

## Research basis

- [ICAO Doc 9303, Part 4](https://www.icao.int/sites/default/files/publications/DocSeries/9303_p4_cons_en.pdf) is the operational source for the detector. It defines the passport biodata page, makes the holder portrait and MRZ mandatory, specifies a two-line 44-character TD3 MRZ, and requires the first MRZ character to be `P` for a machine-readable passport.
- [An Overview of Electronic Passport Security Features](https://link.springer.com/chapter/10.1007/978-3-642-03315-5_11) covers e-passport chip and authentication security. It supports the document context, but does not define OCR strings for page classification.
- [Gemalto passport security report](https://www.id4africa.com/2019/almanac/GEMALTO-Joseph-Leibenguth.pdf) discusses the visual zone, MRZ, portrait substitution, and duplicated personalization data. It supports combining several signals instead of trusting one generic label.
- [Passport photo requirements](https://eoivienna.gov.in/public_files/assets/pdf/Passport_Photo_Requirements_230125.pdf) concern portrait composition/quality rather than a unique OCR text pattern.

**Important limitation:** this is a text-only classifier. A positive result means that the image is very likely a passport biodata page, which normally contains the passport portrait. It does not inspect pixels or prove that a face is visible. For literal portrait verification, add a face detector after this text filter.

The decision rule is deliberately conservative: an ICAO-style passport MRZ is sufficient by itself; otherwise the page needs a passport-specific word/number label plus several independent biodata labels.

In [2]:
from pathlib import Path
import re
import unicodedata

import pandas as pd

INPUT_CSV = Path("ocr_8_angles_partial.csv")
OUTPUT_CSV = Path("passport_detection_results.csv")

TEXT_COLUMNS = [
    "text_angle_000",
    "text_angle_045",
    "text_angle_090",
    "text_angle_135",
    "text_angle_180",
    "text_angle_225",
    "text_angle_270",
    "text_angle_315",
]
REQUIRED_COLUMNS = ["image_name", *TEXT_COLUMNS]

## Small regex pattern database

Weights make MRZ evidence dominant. Generic labels have low weights because they also occur on identity cards, visas, forms, and ordinary documents. Exact patterns are retained as data in a DataFrame so they are easy to review and extend.

In [3]:
PATTERNS = [
    {
        "name": "td3_mrz_pair_strict",
        "target": "mrz_stream",
        "regex": r"P[A-Z0-9<][A-Z<]{3}[A-Z<]{39}[A-Z0-9<]{9}[0-9<][A-Z<]{3}[0-9]{6}[0-9<][MFX<][0-9]{6}[0-9<][A-Z0-9<]{14}[0-9<]{2}",
        "weight": 12,
        "kind": "mrz",
        "description": "Strict contiguous pair of ICAO TD3 passport MRZ lines (44 + 44 characters).",
    },
    {
        "name": "td3_mrz_upper_line",
        "target": "mrz_lines",
        "regex": r"(?m)^P[A-Z0-9<][A-Z<]{3}[A-Z<]{39}$",
        "weight": 8,
        "kind": "mrz",
        "description": "Exact 44-character upper TD3 line; P is the passport document code.",
    },
    {
        "name": "td3_mrz_lower_line",
        "target": "mrz_lines",
        "regex": r"(?m)^[A-Z0-9<]{9}[0-9<][A-Z<]{3}[0-9]{6}[0-9<][MFX<][0-9]{6}[0-9<][A-Z0-9<]{14}[0-9<]{2}$",
        "weight": 6,
        "kind": "mrz",
        "description": "Exact 44-character lower TD3 line with document number, nationality, dates, sex and check digits.",
    },
    {
        "name": "td3_mrz_lower_line_ocr_tolerant",
        "target": "mrz_lines",
        "regex": r"(?m)^[A-Z0-9<]{9}[0-9OIL<][A-Z<]{3}[0-9OIL]{6}[0-9OIL<][MFX<][0-9OIL]{6}[0-9OIL<][A-Z0-9<]{14}[0-9OIL<]{2}$",
        "weight": 4,
        "kind": "mrz",
        "description": "Lower TD3 line allowing common OCR digit confusions O/0 and I/L/1.",
    },
    {
        "name": "td3_mrz_upper_fragment",
        "target": "mrz_lines",
        "regex": r"(?m)^(?=.{36,46}$)P[A-Z0-9<][A-Z<]{3}[A-Z<]{2,34}<<[A-Z<]{1,34}<*$",
        "weight": 5,
        "kind": "mrz",
        "description": "OCR-tolerant upper passport MRZ shape with the primary/secondary name separator <<.",
    },
    {
        "name": "passport_word_exact",
        "target": "text_flat",
        "regex": r"\b(?:PASSPORT|PASSEPORT|PASAPORTE)\b",
        "weight": 4,
        "kind": "passport_label",
        "description": "ICAO VIZ passport heading in English, French or Spanish.",
    },
    {
        "name": "passport_word_ocr_tolerant",
        "target": "text_flat",
        "regex": r"\b(?:PAS5PORT|PA5SPORT|PASSP0RT|PASSEP0RT|PASAP0RTE)\b",
        "weight": 3,
        "kind": "passport_label",
        "description": "Common OCR substitutions in the passport heading.",
    },
    {
        "name": "passport_number_label",
        "target": "text_flat",
        "regex": r"\b(?:PASSPORT|PASSEPORT|PASAPORTE|DOCUMENT)\s*(?:NO|NUMBER|NUMERO|N[O0°º])\b",
        "weight": 3,
        "kind": "passport_label",
        "description": "Passport/document number caption.",
    },
    {
        "name": "surname_label",
        "target": "text_flat",
        "regex": r"\b(?:SURNAME|FAMILY\s+NAME|NOM|APELLIDOS?)\b",
        "weight": 1,
        "kind": "viz_field",
        "description": "Primary identifier/surname caption.",
    },
    {
        "name": "given_names_label",
        "target": "text_flat",
        "regex": r"\b(?:GIVEN\s+NAMES?|FIRST\s+NAMES?|PRENOMS?|NOMBRES?)\b",
        "weight": 1,
        "kind": "viz_field",
        "description": "Secondary identifier/given names caption.",
    },
    {
        "name": "nationality_label",
        "target": "text_flat",
        "regex": r"\b(?:NATIONALITY|NATIONALITE|NACIONALIDAD)\b",
        "weight": 1,
        "kind": "viz_field",
        "description": "Nationality caption.",
    },
    {
        "name": "date_of_birth_label",
        "target": "text_flat",
        "regex": r"\b(?:DATE\s+OF\s+BIRTH|DATE\s+DE\s+NAISSANCE|FECHA\s+DE\s+NACIMIENTO)\b",
        "weight": 1,
        "kind": "viz_field",
        "description": "Date of birth caption.",
    },
    {
        "name": "date_of_expiry_label",
        "target": "text_flat",
        "regex": r"\b(?:DATE\s+OF\s+EXPIRY|DATE\s+OF\s+EXPIRATION|DATE\s+D[' ]?EXPIRATION|FECHA\s+DE\s+(?:EXPIRACION|CADUCIDAD)|VALID\s+UNTIL)\b",
        "weight": 2,
        "kind": "viz_field",
        "description": "Expiry/valid-until caption.",
    },
    {
        "name": "place_of_birth_label",
        "target": "text_flat",
        "regex": r"\b(?:PLACE\s+OF\s+BIRTH|LIEU\s+DE\s+NAISSANCE|LUGAR\s+DE\s+NACIMIENTO)\b",
        "weight": 1,
        "kind": "viz_field",
        "description": "Place of birth caption.",
    },
    {
        "name": "issuing_authority_label",
        "target": "text_flat",
        "regex": r"\b(?:ISSUING\s+AUTHORITY|AUTHORITY|AUTORITE|AUTORIDAD)\b",
        "weight": 1,
        "kind": "viz_field",
        "description": "Issuing authority caption.",
    },
    {
        "name": "holder_signature_label",
        "target": "text_flat",
        "regex": r"\b(?:HOLDER[' ]?S\s+SIGNATURE|SIGNATURE\s+OF\s+HOLDER|SIGNATURE\s+DU\s+TITULAIRE|FIRMA\s+DEL\s+TITULAR)\b",
        "weight": 1,
        "kind": "viz_field",
        "description": "Holder signature caption.",
    },
]

patterns_df = pd.DataFrame(PATTERNS)
patterns_df[["name", "target", "weight", "kind", "regex", "description"]]

,name,target,weight,kind,regex,description
0,td3_mrz_pair_strict,mrz_stream,12,mrz,P[A-Z0-9<][A-Z<]{3}[A-Z<]{39}[A-Z0-9<]{9}[0-9<...,Strict contiguous pair of ICAO TD3 passport MR...
1,td3_mrz_upper_line,mrz_lines,8,mrz,(?m)^P[A-Z0-9<][A-Z<]{3}[A-Z<]{39}$,Exact 44-character upper TD3 line; P is the pa...
2,td3_mrz_lower_line,mrz_lines,6,mrz,(?m)^[A-Z0-9<]{9}[0-9<][A-Z<]{3}[0-9]{6}[0-9<]...,Exact 44-character lower TD3 line with documen...
3,td3_mrz_lower_line_ocr_tolerant,mrz_lines,4,mrz,(?m)^[A-Z0-9<]{9}[0-9OIL<][A-Z<]{3}[0-9OIL]{6}...,Lower TD3 line allowing common OCR digit confu...
4,td3_mrz_upper_fragment,mrz_lines,5,mrz,"(?m)^(?=.{36,46}$)P[A-Z0-9<][A-Z<]{3}[A-Z<]{2,...",OCR-tolerant upper passport MRZ shape with the...
5,passport_word_exact,text_flat,4,passport_label,\b(?:PASSPORT|PASSEPORT|PASAPORTE)\b,"ICAO VIZ passport heading in English, French o..."
6,passport_word_ocr_tolerant,text_flat,3,passport_label,\b(?:PAS5PORT|PA5SPORT|PASSP0RT|PASSEP0RT|PASA...,Common OCR substitutions in the passport heading.
7,passport_number_label,text_flat,3,passport_label,\b(?:PASSPORT|PASSEPORT|PASAPORTE|DOCUMENT)\s*...,Passport/document number caption.
8,surname_label,text_flat,1,viz_field,\b(?:SURNAME|FAMILY\s+NAME|NOM|APELLIDOS?)\b,Primary identifier/surname caption.
9,given_names_label,text_flat,1,viz_field,\b(?:GIVEN\s+NAMES?|FIRST\s+NAMES?|PRENOMS?|NO...,Secondary identifier/given names caption.


## Normalization and matching

All eight angles are searched. A pattern contributes its weight at most once per image, even if it appears in several angles. Unicode accents are removed for multilingual Latin labels, and MRZ lines are compacted while preserving the ICAO filler `<`.

In [4]:
FILLER_TRANSLATION = str.maketrans({
    "«": "<", "‹": "<", "＜": "<", "〈": "<", "⟨": "<"
})

def normalize_ocr(value):
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.upper().translate(FILLER_TRANSLATION)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    return text

def build_search_targets(value):
    text = normalize_ocr(value)
    text_flat = re.sub(r"\s+", " ", text).strip()

    # Remove spaces/punctuation introduced inside OCR-B lines. Restrict candidate
    # lengths to avoid treating ordinary short text lines as MRZ lines.
    mrz_candidates = []
    for line in text.splitlines():
        compact = re.sub(r"[^A-Z0-9<]", "", line)
        if 30 <= len(compact) <= 50:
            mrz_candidates.append(compact)

    return {
        "text_flat": text_flat,
        "mrz_lines": "\n".join(mrz_candidates),
        # The stream also catches OCR output where both MRZ lines were returned
        # without a newline. A full 88-character structural match is required.
        "mrz_stream": re.sub(r"[^A-Z0-9<]", "", text),
    }

COMPILED_PATTERNS = {
    item["name"]: re.compile(item["regex"]) for item in PATTERNS
}
PATTERN_BY_NAME = {item["name"]: item for item in PATTERNS}
PATTERN_ORDER = [item["name"] for item in PATTERNS]
FIELD_PATTERN_NAMES = {
    item["name"] for item in PATTERNS if item["kind"] == "viz_field"
}

def match_one_text(value):
    targets = build_search_targets(value)
    matched = {
        item["name"]
        for item in PATTERNS
        if COMPILED_PATTERNS[item["name"]].search(targets[item["target"]])
    }

    # Do not double-count tolerant fallbacks when the exact pattern matched.
    if "td3_mrz_lower_line" in matched:
        matched.discard("td3_mrz_lower_line_ocr_tolerant")
    if "passport_word_exact" in matched:
        matched.discard("passport_word_ocr_tolerant")
    return matched

def classify_row(row):
    matched_angles = {}
    all_matches = set()

    for column in TEXT_COLUMNS:
        angle_matches = match_one_text(row[column])
        if angle_matches:
            matched_angles[column] = sorted(angle_matches)
            all_matches.update(angle_matches)

    # Apply exact-over-tolerant de-duplication again across angles.
    if "td3_mrz_lower_line" in all_matches:
        all_matches.discard("td3_mrz_lower_line_ocr_tolerant")
    if "passport_word_exact" in all_matches:
        all_matches.discard("passport_word_ocr_tolerant")

    score = sum(PATTERN_BY_NAME[name]["weight"] for name in all_matches)
    field_count = len(all_matches & FIELD_PATTERN_NAMES)
    passport_word = bool(all_matches & {"passport_word_exact", "passport_word_ocr_tolerant"})
    strict_passport_mrz = bool(all_matches & {"td3_mrz_pair_strict", "td3_mrz_upper_line"})
    upper_mrz = bool(all_matches & {"td3_mrz_upper_line", "td3_mrz_upper_fragment"})
    lower_mrz = bool(all_matches & {"td3_mrz_lower_line", "td3_mrz_lower_line_ocr_tolerant"})

    is_passport = (
        strict_passport_mrz
        or (upper_mrz and lower_mrz)
        or (passport_word and lower_mrz)
        or (passport_word and field_count >= 3 and score >= 7)
        or ("passport_number_label" in all_matches and field_count >= 4 and score >= 8)
    )

    ordered_matches = [name for name in PATTERN_ORDER if name in all_matches]
    return pd.Series({
        "matched_pattern_names": "; ".join(ordered_matches),
        "verdict": "YES" if is_passport else "NO",
        "score": score,
        "matched_field_count": field_count,
        "matched_angles": matched_angles,
    })

## Lightweight tests

These tests cover an ICAO example, a mere mention of the word passport, a generic identity-card-like label set, and a VIZ-only passport page. They do not use the input CSV.

In [5]:
def test_row(text):
    row = {column: "" for column in TEXT_COLUMNS}
    row["text_angle_000"] = text
    return pd.Series(row)

test_cases = [
    (
        "ICAO TD3 example",
        "PPUTOERIKSSON<<ANNA<MARIA<<<<<<<<<<<<<<<<<<<\n"
        "L898902C36UTO7408122F3404159ZE184226B<<<<<16",
        "YES",
    ),
    ("ordinary sentence", "Please attach a copy of your passport.", "NO"),
    (
        "generic identity labels",
        "Identity card Surname Given names Nationality Date of birth Date of expiry",
        "NO",
    ),
    (
        "VIZ-only passport evidence",
        "PASSPORT Passport No AB123456 Surname DOE Given names JANE "
        "Nationality CANADIAN Date of birth 01 JAN 1990 Date of expiry 01 JAN 2030",
        "YES",
    ),
]

test_results = []
for name, text, expected in test_cases:
    actual = classify_row(test_row(text))["verdict"]
    test_results.append({"test": name, "expected": expected, "actual": actual})

test_results_df = pd.DataFrame(test_results)
assert (test_results_df["expected"] == test_results_df["actual"]).all(), test_results_df
test_results_df

,test,expected,actual
0,ICAO TD3 example,YES,YES
1,ordinary sentence,NO,NO
2,generic identity labels,NO,NO
3,VIZ-only passport evidence,YES,YES


## Load and validate `ocr_8_angles.csv`

In [6]:
if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"{INPUT_CSV} was not found. Put it in the same directory as this notebook "
        "or change INPUT_CSV in the configuration cell."
    )

ocr_df = pd.read_csv(INPUT_CSV)
missing_columns = [column for column in REQUIRED_COLUMNS if column not in ocr_df.columns]
if missing_columns:
    raise ValueError(f"Missing required CSV columns: {missing_columns}")

ocr_df[TEXT_COLUMNS] = ocr_df[TEXT_COLUMNS].fillna("").astype(str)
print(f"Loaded {len(ocr_df):,} images from {INPUT_CSV}")
ocr_df.head(3)

Loaded 63 images from ocr_8_angles_partial.csv


,image_name,text_angle_000,text_angle_045,text_angle_090,text_angle_135,text_angle_180,text_angle_225,text_angle_270,text_angle_315
0,PDF_Edge_Case_11_page_1.jpg,Random document\nresult simple system model ex...,Random document\nresult simple system model ex...,ejdis eqnu Jeqinu wopuri e6ed sseooud insei ej...,1olabed\nlueinoop eidexe sseooud a! ueinoop Je...,1↓olabed\nsseooud jepow ueunoop enjeΛ efed efe...,1Jolabed\nsseooud jepow quawnoop enjeΛ ebed eb...,value example system process page page value d...,Random document\nresult simple system model ex...
1,PDF_Edge_Case_12_page_1.jpg,Random document\nresult report model number ra...,Random document\nresult report model number ra...,lueinoop e6ed ejduls enje elsßs weisAs efed wo...,LJ0l36ed\neneA ssewoid ssewoud rrp wopurl rep ...,L Jolabed\nJequnu jueunoop e! enjeΛ eßed edmex...,L J0labed\nJeqwnu juewnoop el! enje ebed ejdme...,data number number simple file example page va...,Random document\nresult report model number ra...
2,PDF_Edge_Case_12_page_2.jpg,Random document\nsimple process number simple ...,Random document\nsimple process number simple ...,es<s eisAs Jeqinu Jeqinu lepo quewnoop Jequnu ...,LJ0乙36ed\nebed ueunoop wesAs eidls ejds equnu ...,L J0己abed\neqwnu opuel ejdis jepow eneΛ sseoou...,L J0己aed\nJequnu wopurl edls jepow enpeA sse0o...,model value page text process value model simp...,Random document\nsimple process number simple ...


## Classify images

`result_df` is the requested three-column output. `diagnostics_df` retains scores and angle-level matches for threshold tuning and error analysis.

In [7]:
classification_df = ocr_df.apply(classify_row, axis=1)

result_df = pd.concat(
    [
        ocr_df[["image_name"]].reset_index(drop=True),
        classification_df[["matched_pattern_names", "verdict"]].reset_index(drop=True),
    ],
    axis=1,
)

diagnostics_df = pd.concat(
    [
        ocr_df[["image_name"]].reset_index(drop=True),
        classification_df.reset_index(drop=True),
    ],
    axis=1,
)

result_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(result_df):,} classifications to {OUTPUT_CSV}")
print(result_df["verdict"].value_counts(dropna=False))
result_df

Saved 63 classifications to passport_detection_results.csv
verdict
NO    63
Name: count, dtype: int64


,image_name,matched_pattern_names,verdict
0,PDF_Edge_Case_11_page_1.jpg,passport_number_label,NO
1,PDF_Edge_Case_12_page_1.jpg,passport_number_label,NO
2,PDF_Edge_Case_12_page_2.jpg,passport_number_label,NO
3,PDF_Edge_Case_12_page_3.jpg,passport_number_label,NO
4,PDF_Edge_Case_12_page_4.jpg,passport_number_label,NO
...,...,...,...
58,PDF_Normal_Doc_7_page_7.jpg,passport_number_label,NO
59,PDF_Normal_Doc_8_page_1.jpg,passport_number_label,NO
60,PDF_Normal_Doc_8_page_10.jpg,passport_number_label,NO
61,PDF_Normal_Doc_8_page_2.jpg,passport_number_label,NO


## Inspect borderline and positive cases (optional)

Use this view to review false positives/negatives on labeled data before changing the thresholds. A low-scoring `NO` with several fields is a useful borderline sample; a positive MRZ match is much stronger.

In [8]:
diagnostics_df.sort_values(
    ["verdict", "score"], ascending=[True, False]
).head(25)

,image_name,matched_pattern_names,verdict,score,matched_field_count,matched_angles
26,PDF_Normal_Doc_1_page_3.jpg,passport_word_exact; passport_number_label,NO,7,0,"{'text_angle_000': ['passport_number_label'], ..."
28,PDF_Normal_Doc_2_page_1.jpg,passport_word_exact; passport_number_label,NO,7,0,"{'text_angle_000': ['passport_number_label'], ..."
49,PDF_Normal_Doc_5_page_1.jpg,passport_word_exact; passport_number_label,NO,7,0,"{'text_angle_000': ['passport_number_label'], ..."
0,PDF_Edge_Case_11_page_1.jpg,passport_number_label,NO,3,0,"{'text_angle_000': ['passport_number_label'], ..."
1,PDF_Edge_Case_12_page_1.jpg,passport_number_label,NO,3,0,"{'text_angle_000': ['passport_number_label'], ..."
2,PDF_Edge_Case_12_page_2.jpg,passport_number_label,NO,3,0,"{'text_angle_000': ['passport_number_label'], ..."
3,PDF_Edge_Case_12_page_3.jpg,passport_number_label,NO,3,0,"{'text_angle_000': ['passport_number_label'], ..."
4,PDF_Edge_Case_12_page_4.jpg,passport_number_label,NO,3,0,"{'text_angle_000': ['passport_number_label'], ..."
5,PDF_Edge_Case_12_page_5.jpg,passport_number_label,NO,3,0,"{'text_angle_000': ['passport_number_label'], ..."
6,PDF_Edge_Case_12_page_6.jpg,passport_number_label,NO,3,0,"{'text_angle_000': ['passport_number_label'], ..."
